In [14]:
import json
import math
import os

AMPDAMP_REPORT_DIR = "reports/8Q-heisen-ampdamp"
DEPH_REPORT_DIR = "reports/8Q-heisen-deph"
OUTPUT_DIR = "reports"


def fmt_mean_std(mean, std):
    return f"${mean:.3f} \\pm {std:.3f}$"


def fmt_error_rate(err):
    return f"{err:.3f}"


def fmt_sci(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_overhead(value, precision=3):
    exponent = math.floor(math.log10(abs(value)))
    mantissa = value / (10 ** exponent)
    return f"${mantissa:.{precision}f} \\times 10^{{{exponent}}}$"


def fmt_int(value):
    return f"{int(round(value))}"


def load_export(path):
    with open(path) as f:
        data = json.load(f)
    rows_by_order = {int(r["order"]): r for r in data["rows"]}
    return data, rows_by_order


def build_noise_free_row(ad_data):
    mean = ad_data["noise_free_mean"]
    std = ad_data["noise_free_std"]
    return (
        r"\textbf{Noise-free VQE estimation} & -- & "
        f"{fmt_mean_std(mean, std)} & -- & -- & -- & -- & -- & -- \\\\"
    )


def build_unmitigated_block(deph_data, ad_data):
    return [
        r"\multirow{2}{*}{\textbf{Unmitigated VQE estimation}} & Dephasing & "
        f"{fmt_mean_std(deph_data['unmitigated_mean'], deph_data['unmitigated_std'])} "
        r"& -- & -- & -- & -- & -- & -- \\",
        r" & AD & "
        f"{fmt_mean_std(ad_data['unmitigated_mean'], ad_data['unmitigated_std'])} "
        r"& -- & -- & -- & -- & -- & -- \\",
    ]


def build_zne_block(order, d, a):
    for field in ("depth", "gate", "overhead"):
        if not math.isclose(d[field], a[field], rel_tol=1e-9):
            raise ValueError(
                f"Order {order}: '{field}' differs between Dephasing "
                f"({d[field]}) and AD ({a[field]}) — expected identical "
                f"values for this classical quantity."
            )

    overhead_str = fmt_overhead(d["overhead"])
    depth_str = fmt_int(d["depth"])
    gate_str = fmt_int(d["gate"])

    return [
        r"\multirow{2}{*}{\textbf{ZNE of order " + str(order) + r"}} & "
        r"Dephasing & "
        f"{fmt_mean_std(d['zne_mean'], d['zne_std'])} & "
        f"{fmt_error_rate(d['error_rate'])} & "
        f"{fmt_sci(d['runtime'])} & "
        f"{fmt_sci(d['comp_cost'])} & "
        r"\multirow{2}{*}{" + overhead_str + r"} & "
        r"\multirow{2}{*}{" + depth_str + r"} & "
        r"\multirow{2}{*}{" + gate_str + r"} \\",
        r" & AD & "
        f"{fmt_mean_std(a['zne_mean'], a['zne_std'])} & "
        f"{fmt_error_rate(a['error_rate'])} & "
        f"{fmt_sci(a['runtime'])} & "
        f"{fmt_sci(a['comp_cost'])} & & & \\\\",
    ]


def build_table_body(deph_path, ad_path):
    deph_data, deph_rows = load_export(deph_path)
    ad_data, ad_rows = load_export(ad_path)

    orders = sorted(set(deph_rows) & set(ad_rows))

    lines = [r"\hline"]
    lines.append(build_noise_free_row(ad_data))
    lines.append(r"\hline")
    lines.extend(build_unmitigated_block(deph_data, ad_data))

    for order in orders:
        lines.append(r"\hline")
        lines.extend(build_zne_block(order, deph_rows[order], ad_rows[order]))

    lines.append(r"\hline")
    return "\n".join(lines)


deph_path = os.path.join(DEPH_REPORT_DIR, "zne_multivar_dephasing.json")
ad_path = os.path.join(AMPDAMP_REPORT_DIR, "zne_multivar_ampdamp.json")

body = build_table_body(deph_path, ad_path)

out_path = os.path.join(OUTPUT_DIR, "zne_multivar_table_body.tex")
with open(out_path, "w") as f:
    f.write(body + "\n")

print(body)
print(f"\nWritten to {out_path}")

\hline
\textbf{Noise-free VQE estimation} & -- & $-13.198 \pm 0.052$ & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{*}{\textbf{Unmitigated VQE estimation}} & Dephasing & $-10.067 \pm 0.116$ & -- & -- & -- & -- & -- & -- \\
 & AD & $-11.592 \pm 0.131$ & -- & -- & -- & -- & -- & -- \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 1}} & Dephasing & $-12.289 \pm 0.087$ & 0.909 & $1.876 \times 10^{12}$ & $1.383 \times 10^{15}$ & \multirow{2}{*}{$1.225 \times 10^{1}$} & \multirow{2}{*}{70} & \multirow{2}{*}{174} \\
 & AD & $-13.008 \pm 0.076$ & 0.190 & $1.671 \times 10^{12}$ & $1.219 \times 10^{15}$ & & & \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 2}} & Dephasing & $-12.856 \pm 0.062$ & 0.342 & $8.037 \times 10^{13}$ & $6.628 \times 10^{15}$ & \multirow{2}{*}{$1.658 \times 10^{2}$} & \multirow{2}{*}{236} & \multirow{2}{*}{616} \\
 & AD & $-13.134 \pm 0.055$ & 0.064 & $7.160 \times 10^{13}$ & $5.839 \times 10^{15}$ & & & \\
\hline
\multirow{2}{*}{\textbf{ZNE of order 3}} & Dephasing &